# Visualize motion paths of multiple recordings

<video controls src="./assets/path_seeking_recording_multi.webm">

Load the nanotube + methane example recordings as MDAnalsis universes:

In [1]:
from nanover.mdanalysis import universe_from_recording

universes = [
    universe_from_recording("nanotube-methane-01.nanover.zip"),
    universe_from_recording("nanotube-methane-02.nanover.zip"),
    universe_from_recording("nanotube-methane-03.nanover.zip"),
]

Set up the server with playback of the universes:

In [2]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulations = []
for universe in universes:
    simulation = UniverseSimulation.from_universe(universe)
    simulation.playback_factor = 30
    simulation.load()
    simulations.append(simulation)

imd_runner = OmniRunner.with_basic_server(*simulations, port=0, name="EXAMPLE: trajectory seeker (multi)")

Import the jupyter utilities for drawing + interaction:

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Define a FrameListener to constantly update path alignment as structure moves during playback:

In [4]:
from nanover.utilities.transforms import StructureAlignment
from nanover.jupyter.alignment_transform import AlignmentTransform

STRUCTURE_ATOMS = universes[0].atoms[range(0, 60)]
alignment = StructureAlignment.from_atom_group(STRUCTURE_ATOMS)

alignment_update = AlignmentTransform.from_runner(imd_runner)
alignment_update.config(key="structure", alignment=alignment)
alignment_update.start()

Calculate molecule centroids in each frame and transform them into a common reference frame:

In [5]:
import numpy as np

MOLECULE_PATHS = []
for universe in universes:
    atoms = universe.atoms[range(60, 65)]
    centroids = []
    for i, _ in enumerate(universe.trajectory):
        positions = atoms.positions / 10  # angstrom -> nm
        centroid = np.mean(positions, axis=0)
        transform_i = alignment.calculate_transform_to_universe(universe)
        centroids.append(transform_i.point_parent_to_local(centroid))
    MOLECULE_PATHS.append(centroids)

Add a line to the scene that shows the path of the centroid over time:

In [6]:
from nanover.recording.playback import SCENE_POSE_IDENTITY
from nanover.utilities.change_buffers import DictionaryChange

ACTIVE_UNIVERSE_INDEX = 0

def redraw_paths():
    for i, positions in enumerate(MOLECULE_PATHS):
        color = [1.0, 0.0, 0.0, 1.0] if i == ACTIVE_UNIVERSE_INDEX else [1.0, 1.0, 1.0, 1.0]
        utilities.objects.update_line(f"path.{i}", positions=positions, size=0.01, color=color, parent="structure")

def select_universe(index):
    global ACTIVE_UNIVERSE_INDEX
    ACTIVE_UNIVERSE_INDEX = index % len(universes)
    redraw_paths()
    scene = imd_runner.app_server.copy_state().get("scene", SCENE_POSE_IDENTITY)
    imd_runner.load(ACTIVE_UNIVERSE_INDEX)
    imd_runner.app_server.update_state(DictionaryChange(updates={"scene":scene}))

select_universe(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Define a mode that shows the nearest point on the centroid path as the user controllers hover close to it, and seeks the recording when they click:

In [7]:
from nanover.jupyter import Mode
from nanover.utilities.intersection import closest_point_on_polyline


class SeekLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "secondary":
            select_universe(ACTIVE_UNIVERSE_INDEX + 1)
            return
        elif button != "primary":
            return

        universe = universes[ACTIVE_UNIVERSE_INDEX]
        centroids = MOLECULE_PATHS[ACTIVE_UNIVERSE_INDEX]
        simulation = simulations[ACTIVE_UNIVERSE_INDEX]

        pos_world = cursor["position"]
        pos_scene = utilities.scene_transform.point_parent_to_local(pos_world)
        pos_align = alignment.calculate_transform_to_universe(universe).point_parent_to_local(pos_scene)

        result = closest_point_on_polyline(centroids, pos_align)
        utilities.notify_all(f"seek to frame #{result.index}")
        simulation.seek_to_frame_index(result.index)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        universe = universes[ACTIVE_UNIVERSE_INDEX]
        centroids = MOLECULE_PATHS[ACTIVE_UNIVERSE_INDEX]

        pos_world = cursor["position"]
        pos_scene = utilities.scene_transform.point_parent_to_local(pos_world)
        pos_align = alignment.calculate_transform_to_universe(universe).point_parent_to_local(pos_scene)

        result = closest_point_on_polyline(centroids, pos_align)

        if result.distance < 0.25:
            utilities.objects.update_shape(f"cursor.{key}", position=result.point, size=0.05, parent="structure")
        else:
            utilities.objects.remove_shape(f"cursor.{key}")


utilities.use_interaction_modes()
utilities.add_interaction_mode(SeekLineMode, "seek", icon="⏩")

Start playback running:

In [8]:
imd_runner.load(0)